# Lab 6b: WordCount with RDDs, DataFrames and SQL

Run the cells one at a time, from the top. The same notebook runs in the development container and in Google Colab: the first cell detects where it is.

## Set up the environment

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/gdrive")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"],
                   check=True,
                   stdout=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark==4.1.1"], check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    DATA = "/content/gdrive/MyDrive/big-data-labs/datasets"
    OUTPUT = "/content/gdrive/MyDrive/big-data-labs/output/lab06b"
else:
    DATA = "../../datasets"
    OUTPUT = "../shared/output/lab06b"

print("Google Colab" if IN_COLAB else "Development container", "| data in", DATA)

## Start Spark

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder.master("local[*]").appName("lab06b-wordcount")
         .config("spark.sql.warehouse.dir", f"{OUTPUT}/spark-warehouse")
         .getOrCreate())
sc = spark.sparkContext
print("Spark", spark.version, "| cores:", sc.defaultParallelism)

## Read the books

`textFile` reads the files that match the pattern and gives an RDD with one element per line. Reading is lazy: nothing is read until an action, such as `take` or `count`, needs the data.

In [ ]:
lines = sc.textFile(f"{DATA}/gutenberg-en/*.txt")
print("Partitions:", lines.getNumPartitions())
for line in lines.take(5):
    print(repr(line))

In [ ]:
lines.count()

## WordCount step by step

Each step is a transformation that gives a new RDD. `take` is an action: it runs the steps that lead to it and brings a few elements to the notebook.

In [ ]:
tokens = lines.flatMap(lambda line: line.split())
tokens.take(10)

In [ ]:
pairs = tokens.map(lambda word: (word, 1))
pairs.take(5)

In [ ]:
from operator import add

counts = pairs.reduceByKey(add)
counts.take(5)

In [ ]:
counts.sortBy(lambda pair: -pair[1]).take(10)

## Clean the words

Splitting on spaces keeps the case and the punctuation: `The`, `the` and `the,` are three different words. The function `words` turns the line into lowercase, changes every character that is not a letter into a space and then splits on the spaces.

In [ ]:
def words(line):
    letters = "".join(char if char.isalpha() else " " for char in line.lower())
    return letters.split()


words("The whale, the WHALE! 1851")

In [ ]:
clean_counts = lines.flatMap(words).map(lambda word: (word, 1)).reduceByKey(add)
print("Distinct words:", clean_counts.count())
clean_counts.sortBy(lambda pair: -pair[1]).take(10)

In [ ]:
clean_counts.filter(lambda pair: pair[0] in ("whale", "ahab", "sherlock")).collect()

## The same count with DataFrames

`spark.read.text` gives a DataFrame with one column, `value`, and one row per line. `split` with the regular expression `[^\p{L}]+` cuts the line at everything that is not a letter, and `explode` gives one row per word.

In [ ]:
from pyspark.sql import functions as F

books = spark.read.text(f"{DATA}/gutenberg-en")
df_counts = (books.select(F.explode(F.split(F.lower("value"), r"[^\p{L}]+")).alias("word"))
             .filter(F.col("word") != "")
             .groupBy("word")
             .count()
             .orderBy(F.desc("count"), "word"))
df_counts.show(10)

Both versions give the same count, word by word:

In [ ]:
sorted(clean_counts.collect()) == sorted(tuple(row) for row in df_counts.collect())

The plan of the DataFrame has an `Exchange`: the shuffle that brings all the rows of a word to the same partition, as `reduceByKey` does with the RDD.

In [ ]:
df_counts.explain()

## The same count in Spark SQL

In [ ]:
books.createOrReplaceTempView("lines")
spark.sql("""
    SELECT word, count(*) AS count
    FROM (SELECT explode(split(lower(value), '[^\\\\p{L}]+')) AS word FROM lines)
    WHERE word <> ''
    GROUP BY word
    ORDER BY count DESC, word
""").show(10)

## The Spanish books

In [ ]:
spanish = sc.textFile(f"{DATA}/gutenberg-es/*.txt")
spanish_counts = spanish.flatMap(words).map(lambda word: (word, 1)).reduceByKey(add)
spanish_counts.sortBy(lambda pair: -pair[1]).take(10)

## Save the count

In [ ]:
(df_counts.coalesce(1).write.mode("overwrite").csv(f"{OUTPUT}/wordcount-en", header=True))
[name for name in sorted(os.listdir(f"{OUTPUT}/wordcount-en")) if not name.startswith(".")]

## Stop Spark

In [ ]:
spark.stop()